# 🔬 Auditoría Avanzada de Valores "Unknown"

**Proyecto**: Marketing Analytics para Empresa Fintech — TFM  
**Objetivo**: Determinar si los valores `"unknown"` en el dataset contienen información predictiva, identificar patrones y definir la estrategia óptima de tratamiento para lograr un CSV perfectamente limpio.  
**Dataset**: `bank-additional_bank-additional-full.csv` (41,188 × 21)  
**Autor**: Jesús  
**Fecha**: Marzo 2026  

---

> ⚠️ **Este notebook es de SOLO LECTURA** sobre los datos. No se modifica el CSV fuente.

## 1. Configuración del Entorno

In [ ]:
# ==============================================================================
# IMPORTS
# ==============================================================================
import pandas as pd
import numpy as np
import os

# Visualización
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Estadística
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal

# Configuración
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Entorno configurado correctamente')

## 2. Carga del Dataset

In [ ]:
# ==============================================================================
# CARGA DEL DATASET ORIGINAL (sin modificar)
# ==============================================================================
CSV_PATH = os.path.join('..', '..', 'datos', 'bank-additional_bank-additional-full.csv')
df = pd.read_csv(CSV_PATH, sep=';')

n_rows, n_cols = df.shape
print(f'📂 Dataset cargado: {n_rows:,} filas × {n_cols} columnas')
print(f'💾 Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

# Columnas con unknowns
COLS_UNK = ['job', 'marital', 'education', 'default', 'housing', 'loan']

# Variable target numérica (auxiliar)
df['y_num'] = (df['y'] == 'yes').astype(int)

# Tasa de conversión global
tasa_global = df['y_num'].mean() * 100
print(f'\n🎯 Tasa de conversión global: {tasa_global:.2f}%')
print(f'   yes: {df["y_num"].sum():,} | no: {(1-df["y_num"]).sum():,.0f}')

---

## 3. Diagnóstico de Unknowns — Panorama General

Antes de decidir qué hacer con los unknowns, necesitamos entender:
1. **Cuántos hay** por columna
2. **Dónde co-ocurren** (si un registro tiene unknown en una columna, ¿también en otras?)
3. **Si son informativos** para la variable target

In [ ]:
# ==============================================================================
# 3.1 CONTEO DE UNKNOWNS POR COLUMNA
# ==============================================================================
unknown_report = pd.DataFrame({
    'conteo': [(df[col] == 'unknown').sum() for col in COLS_UNK],
    'porcentaje': [(df[col] == 'unknown').mean() * 100 for col in COLS_UNK]
}, index=COLS_UNK)

unknown_report = unknown_report.sort_values('conteo', ascending=False)
unknown_report['severidad'] = unknown_report['porcentaje'].apply(
    lambda x: '⛔ CRÍTICO' if x > 15 else '⚠️ MODERADO' if x > 5 else '✅ BAJO'
)

print('\n📊 UNKNOWNS POR COLUMNA:')
print('=' * 60)
display(unknown_report)

In [ ]:
# ==============================================================================
# 3.2 VISUALIZACIÓN: Proporción de unknowns
# ==============================================================================
fig = px.bar(
    unknown_report.reset_index(),
    x='index', y='porcentaje',
    color='porcentaje',
    color_continuous_scale='Reds',
    text=unknown_report['conteo'].apply(lambda x: f'{x:,}'),
    labels={'index': 'Columna', 'porcentaje': '% Unknown'},
    title='📊 Proporción de Valores "Unknown" por Columna'
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white', height=450, width=800,
    xaxis_title='', yaxis_title='% de registros con unknown',
    coloraxis_showscale=False
)
fig.add_hline(y=5, line_dash='dash', line_color='orange',
              annotation_text='Umbral moderado (5%)', annotation_position='top right')
fig.add_hline(y=15, line_dash='dash', line_color='red',
              annotation_text='Umbral crítico (15%)', annotation_position='top right')
fig.show()

---

## 4. Co-ocurrencia de Unknowns

¿Los unknowns son independientes o aparecen juntos en las mismas filas?

In [ ]:
# ==============================================================================
# 4.1 MATRIZ DE CO-OCURRENCIA
# ==============================================================================
df_unk_flags = pd.DataFrame()
for col in COLS_UNK:
    df_unk_flags[col] = (df[col] == 'unknown').astype(int)

df_unk_flags['total_unknowns'] = df_unk_flags[COLS_UNK].sum(axis=1)

# Co-ocurrencia
print('\n🔗 REGISTROS POR NÚMERO DE COLUMNAS CON UNKNOWN:')
print('=' * 50)
cooccur = df_unk_flags['total_unknowns'].value_counts().sort_index()
for k, v in cooccur.items():
    bar = '█' * int(v / n_rows * 200)
    print(f'   {k} unknowns: {v:>6,} ({v/n_rows*100:>5.2f}%) {bar}')

# housing + loan siempre juntos?
both = ((df['housing']=='unknown') & (df['loan']=='unknown')).sum()
h_only = ((df['housing']=='unknown') & (df['loan']!='unknown')).sum()
l_only = ((df['housing']!='unknown') & (df['loan']!='unknown')).sum()
print(f'\n   housing+loan ambos unknown: {both:,} (SIEMPRE juntos)')
print(f'   solo housing unknown: {h_only}')
print(f'   solo loan unknown: {l_only - (n_rows - 990):,}')

In [ ]:
# ==============================================================================
# 4.2 HEATMAP DE CO-OCURRENCIA
# ==============================================================================
corr_unk = df_unk_flags[COLS_UNK].corr().round(3)

fig = ff.create_annotated_heatmap(
    z=corr_unk.values,
    x=list(corr_unk.columns),
    y=list(corr_unk.index),
    colorscale='RdBu_r',
    showscale=True
)
fig.update_layout(
    title='🔗 Correlación entre flags de Unknown (1=unknown, 0=known)',
    width=700, height=550
)
fig.show()

---

## 5. ¿Son los Unknowns Informativos? — Test Estadístico

La pregunta central: **¿el hecho de que un valor sea "unknown" nos dice algo sobre si el cliente se va a suscribir?**

Usamos el **test Chi-cuadrado de independencia** (χ²) para determinarlo.

In [ ]:
# ==============================================================================
# 5.1 CHI-CUADRADO: unknown vs target
# ==============================================================================
print('\n🧪 TEST CHI-CUADRADO: ¿Es "unknown" informativo para la target?')
print('=' * 80)
print(f'{"Columna":20s} {"chi²":>10s} {"p-value":>12s} {"Conv. known":>13s} {"Conv. unk":>11s} {"Diferencia":>12s} {"Veredicto":>15s}')
print('-' * 95)

chi2_results = []

for col in COLS_UNK:
    has_unk = (df[col] == 'unknown').sum()
    if has_unk == 0:
        continue
    
    is_unk = df[col] == 'unknown'
    ct = pd.crosstab(is_unk, df['y'])
    chi2, p, dof, expected = chi2_contingency(ct)
    
    rate_known = df.loc[~is_unk, 'y_num'].mean() * 100
    rate_unk = df.loc[is_unk, 'y_num'].mean() * 100
    diff = rate_unk - rate_known
    
    sig = '⛔ PREDICTIVO' if p < 0.001 else '⚠️ Significativo' if p < 0.05 else '✅ No inform.'
    sign = '+' if diff > 0 else ''
    
    chi2_results.append({
        'columna': col, 'chi2': chi2, 'p_value': p,
        'conv_known': rate_known, 'conv_unknown': rate_unk,
        'diff_pp': diff, 'significativo': p < 0.05
    })
    
    print(f'{col:20s} {chi2:>10.2f} {p:>12.6f} {rate_known:>12.2f}% {rate_unk:>10.2f}% {sign}{diff:>11.2f}pp {sig:>15s}')

df_chi2 = pd.DataFrame(chi2_results)

In [ ]:
# ==============================================================================
# 5.2 VISUALIZACIÓN: Tasa de conversión known vs unknown
# ==============================================================================
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Bar(
    name='Known', x=df_chi2['columna'], y=df_chi2['conv_known'],
    marker_color='#2ecc71', text=[f"{v:.1f}%" for v in df_chi2['conv_known']],
    textposition='outside'
))
fig.add_trace(go.Bar(
    name='Unknown', x=df_chi2['columna'], y=df_chi2['conv_unknown'],
    marker_color='#e74c3c', text=[f"{v:.1f}%" for v in df_chi2['conv_unknown']],
    textposition='outside'
))

fig.update_layout(
    title='🎯 Tasa de Conversión: Clientes Known vs Unknown',
    xaxis_title='', yaxis_title='Conversión (%)',
    barmode='group', plot_bgcolor='white',
    height=500, width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.add_hline(y=tasa_global, line_dash='dot', line_color='gray',
              annotation_text=f'Media global: {tasa_global:.1f}%')
fig.show()

---

## 6. Análisis Profundo de `default` (20.87% unknown)

La columna más problemática. Necesitamos entender el **perfil** del cliente con `default=unknown`.

In [ ]:
# ==============================================================================
# 6.1 DISTRIBUCIÓN DE DEFAULT Y CONVERSIÓN
# ==============================================================================
print('\n📊 DISTRIBUCIÓN DE default:')
print('=' * 60)
for val in ['no', 'unknown', 'yes']:
    mask = df['default'] == val
    cnt = mask.sum()
    rate = df.loc[mask, 'y_num'].mean() * 100
    print(f'   {val:10s}: {cnt:>6,} ({cnt/n_rows*100:.2f}%) | conversión = {rate:.2f}%')

print(f'\n   ⚠️  Solo hay 3 registros con default=yes (0.007%)')
print(f'   → Imputar con la moda ("no") sería ESTADÍSTICAMENTE INCORRECTO')
print(f'     porque los unknowns tienen un perfil de conversión MUY diferente a "no"')

In [ ]:
# ==============================================================================
# 6.2 PERFIL COMPARATIVO: default=unknown vs default=no
# ==============================================================================
print('\n👤 PERFIL COMPARATIVO de clientes (default=unknown vs default=no):')
print('=' * 70)

# Variables numéricas
profile_vars = ['age', 'duration', 'campaign', 'previous', 'euribor3m', 'nr.employed']

profile_data = []
for feat in profile_vars:
    vals_unk = df.loc[df['default']=='unknown', feat]
    vals_no = df.loc[df['default']=='no', feat]
    
    # Mann-Whitney U test (no asume normalidad)
    stat, p = mannwhitneyu(vals_unk, vals_no, alternative='two-sided')
    sig = '⛔ Sig.' if p < 0.001 else '⚠️ Sig.' if p < 0.05 else '✅ No sig.'
    
    profile_data.append({
        'variable': feat,
        'mean_unknown': vals_unk.mean(),
        'mean_no': vals_no.mean(),
        'median_unknown': vals_unk.median(),
        'median_no': vals_no.median(),
        'p_value': p,
        'significativo': sig
    })

df_profile = pd.DataFrame(profile_data)
display(df_profile.style.format({
    'mean_unknown': '{:.2f}', 'mean_no': '{:.2f}',
    'median_unknown': '{:.1f}', 'median_no': '{:.1f}',
    'p_value': '{:.6f}'
}))

In [ ]:
# ==============================================================================
# 6.3 DISTRIBUCIÓN DE EDAD: default=unknown vs default=no
# ==============================================================================
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df.loc[df['default']=='no', 'age'], name='default=no',
    opacity=0.7, marker_color='#2ecc71', histnorm='percent', nbinsx=40
))
fig.add_trace(go.Histogram(
    x=df.loc[df['default']=='unknown', 'age'], name='default=unknown',
    opacity=0.7, marker_color='#e74c3c', histnorm='percent', nbinsx=40
))

fig.update_layout(
    title='Distribución de Edad: default=no vs default=unknown',
    xaxis_title='Edad', yaxis_title='% de registros',
    barmode='overlay', plot_bgcolor='white',
    height=450, width=900
)
fig.show()

In [ ]:
# ==============================================================================
# 6.4 EDUCACIÓN POR GRUPOS DE DEFAULT
# ==============================================================================
for dval in ['no', 'unknown']:
    mask = df['default'] == dval
    print(f'\n   default=\'{dval}\' - Distribución de educación:')
    vc = df.loc[mask, 'education'].value_counts()
    for edu, cnt in vc.items():
        pct = cnt / mask.sum() * 100
        bar = '█' * int(pct / 2)
        print(f'     {edu:25s}: {cnt:>5,} ({pct:>5.1f}%) {bar}')

---

## 7. Análisis Profundo de `education` (4.20% unknown)

Segundo unknown estadísticamente significativo.

In [ ]:
# ==============================================================================
# 7.1 CONVERSIÓN POR NIVEL EDUCATIVO (incluyendo unknown)
# ==============================================================================
edu_conv = df.groupby('education').agg(
    total=('y_num', 'count'),
    conversiones=('y_num', 'sum')
).reset_index()
edu_conv['tasa'] = (edu_conv['conversiones'] / edu_conv['total'] * 100).round(2)
edu_conv = edu_conv.sort_values('tasa', ascending=True)

# Colores: resaltar unknown
colors = ['#e74c3c' if v == 'unknown' else '#3498db' for v in edu_conv['education']]

fig = go.Figure(go.Bar(
    y=edu_conv['education'], x=edu_conv['tasa'],
    orientation='h', marker_color=colors,
    text=[f"{t:.1f}% (n={n:,})" for t, n in zip(edu_conv['tasa'], edu_conv['total'])],
    textposition='outside'
))
fig.update_layout(
    title='🎯 Conversión por Nivel Educativo (unknown resaltado en rojo)',
    xaxis_title='Conversión (%)', yaxis_title='',
    plot_bgcolor='white', height=450, width=900
)
fig.add_vline(x=tasa_global, line_dash='dot', line_color='gray',
              annotation_text=f'Media: {tasa_global:.1f}%')
fig.show()

In [ ]:
# ==============================================================================
# 7.2 EDUCATION=UNKNOWN: ¿Qué trabajos tienen?
# ==============================================================================
mask_edu_unk = df['education'] == 'unknown'
print(f'\n🔍 education=unknown ({mask_edu_unk.sum():,} registros) - Top trabajos:')
print('=' * 50)

edu_unk_jobs = df.loc[mask_edu_unk, 'job'].value_counts()
for job, cnt in edu_unk_jobs.items():
    pct = cnt / mask_edu_unk.sum() * 100
    bar = '█' * int(pct / 2)
    print(f'   {job:20s}: {cnt:>4,} ({pct:>5.1f}%) {bar}')

print(f'\n   → blue-collar domina (26.2%) — coincide con perfiles')
print(f'     donde la educación formal puede no estar registrada')

---

## 8. ¿Siguen un Patrón los Unknowns? — Análisis Multivariante

In [ ]:
# ==============================================================================
# 8.1 PATRÓN TEMPORAL: ¿Se concentran los unknowns en ciertos meses?
# ==============================================================================
meses_orden = ['mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

# Porcentaje de unknowns por mes para default y education
temporal = []
for m in meses_orden:
    mask_m = df['month'] == m
    total_m = mask_m.sum()
    if total_m == 0:
        continue
    for col in ['default', 'education']:
        pct_unk = (df.loc[mask_m, col] == 'unknown').sum() / total_m * 100
        temporal.append({'month': m, 'variable': col, 'pct_unknown': pct_unk})

df_temporal = pd.DataFrame(temporal)

fig = px.line(
    df_temporal, x='month', y='pct_unknown', color='variable',
    markers=True, title='📅 % de Unknowns por Mes de Contacto',
    labels={'pct_unknown': '% unknown', 'month': 'Mes'},
    category_orders={'month': meses_orden}
)
fig.update_layout(plot_bgcolor='white', height=400, width=900)
fig.show()

In [ ]:
# ==============================================================================
# 8.2 PATRÓN POR MÉTODO DE CONTACTO
# ==============================================================================
print('\n📞 ¿Los unknowns dependen del método de contacto?')
print('=' * 60)

for col in ['default', 'education', 'job', 'marital']:
    for method in ['cellular', 'telephone']:
        mask = df['contact'] == method
        pct = (df.loc[mask, col] == 'unknown').sum() / mask.sum() * 100
        print(f'   {col:15s} | {method:10s}: {pct:.2f}% unknown')
    print()

In [ ]:
# ==============================================================================
# 8.3 ¿LOS DEFAULT=UNKNOWN SE CONCENTRAN EN CIERTAS FRANJAS DE EDAD?
# ==============================================================================
# Crear bins de edad
df['age_bin'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                       labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

age_unk = df.groupby('age_bin', observed=True).apply(
    lambda g: pd.Series({
        'total': len(g),
        'default_unk_pct': (g['default'] == 'unknown').mean() * 100,
        'education_unk_pct': (g['education'] == 'unknown').mean() * 100,
        'conversion': g['y_num'].mean() * 100
    })
).reset_index()

print('\n📊 Unknowns y conversión por franja de edad:')
print('=' * 80)
print(f'{"Franja":>10s} {"Total":>8s} {"default_unk%":>14s} {"education_unk%":>16s} {"Conversión%":>13s}')
print('-' * 65)
for _, row in age_unk.iterrows():
    print(f'{row["age_bin"]:>10s} {row["total"]:>8,.0f} {row["default_unk_pct"]:>13.1f}% {row["education_unk_pct"]:>15.1f}% {row["conversion"]:>12.1f}%')

In [ ]:
# ==============================================================================
# 8.4 KRUSKAL-WALLIS: ¿Default unknown se diferencia por variables macro?
# ==============================================================================
print('\n🧪 KRUSKAL-WALLIS: ¿Default groups difieren en variables macroeconómicas?')
print('=' * 70)

macro_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in macro_cols:
    groups = [df.loc[df['default']==val, col].values for val in ['no', 'unknown']]
    stat, p = kruskal(*groups)
    sig = '⛔ SIG.' if p < 0.001 else '⚠️ sig.' if p < 0.05 else '✅ no sig.'
    print(f'   {col:20s}: H={stat:>10.2f}, p={p:.6f} {sig}')

---

## 9. Heatmap Cruzado: Conversión por Default × Education

In [ ]:
# ==============================================================================
# 9.1 HEATMAP: Conversión Default × Education
# ==============================================================================
cross = pd.crosstab(
    df['default'], df['education'],
    values=df['y_num'], aggfunc='mean'
) * 100

fig = ff.create_annotated_heatmap(
    z=cross.values.round(1),
    x=list(cross.columns),
    y=list(cross.index),
    colorscale='YlGn', showscale=True
)
fig.update_layout(
    title='Tasa de Conversión (%) — Default × Education',
    width=1000, height=400
)
fig.show()

---

## 10. Veredicto Final y Estrategia de Limpieza

### Resumen de hallazgos:

| Columna | Unknowns | % | ¿Informativo? | Patrón detectado | **Estrategia** |
|:--------|:--------:|:--:|:---:|:---|:---|
| **`default`** | 8,597 | 20.87% | **SÍ** (χ²=405) | Más mayores, menor educación, 7.7pp menos conversión | **MANTENER como categoría** |
| **`education`** | 1,731 | 4.20% | **SÍ** (χ²=18.6) | Dominan blue-collar, +3.4pp conversión | **MANTENER como categoría** |
| `housing` | 990 | 2.40% | No | Siempre co-ocurre con loan | **ELIMINAR filas** |
| `loan` | 990 | 2.40% | No | Siempre co-ocurre con housing | **ELIMINAR filas** |
| `job` | 330 | 0.80% | No | Sin patrón, negligible | **MANTENER como categoría** |
| `marital` | 80 | 0.19% | No | Sin patrón, negligible | **MANTENER como categoría** |

### ¿Por qué NO convertir a NaN ni imputar default/education?

1. **`default=unknown`** tiene **conversión = 5.15%** vs **12.88%** para `default=no`. Son perfiles reales: personas mayores (media 43.4 años) con menor nivel educativo donde el banco no tiene historial crediticio. **El unknown ES la información.**

2. **`education=unknown`** convierte al **14.50%**, por encima de la media global (11.27%). Probablemente son clientes con educación no formal (blue-collar, autónomos) que no declararon su nivel educativo. **Eliminar o imputar destruiría señal predictiva.**

3. Imputar con la moda viola el principio de que **la ausencia de información es información en sí misma** (Missing Not At Random — MNAR).

In [ ]:
# ==============================================================================
# 10.1 RESUMEN EJECUTIVO
# ==============================================================================
print('\n' + '=' * 90)
print('  VEREDICTO FINAL — ESTRATEGIA DE LIMPIEZA')
print('=' * 90)

print('''
🟢 ACCIONES PARA EL CSV LIMPIO:

  1. ELIMINAR 12 filas duplicadas exactas
     → df.drop_duplicates(inplace=True)

  2. ELIMINAR 990 filas con housing+loan=unknown
     → No son informativos (p=0.68)
     → Siempre co-ocurren, bajo impacto (2.4%)

  3. MANTENER default="unknown" como categoría legítima
     → ⛔ NO convertir a NaN (destruye 7.7pp de señal)
     → ⛔ NO imputar con la moda "no" (viola MNAR)
     → ✅ Tratar como una 3ra categoría: no / unknown / yes
     → En encoding: One-Hot con 3 valores

  4. MANTENER education="unknown" como categoría legítima
     → Convierte 3.4pp MÁS que el promedio
     → En encoding: incluir como categoría adicional

  5. MANTENER job="unknown" y marital="unknown"
     → Cantidades negligibles (330 y 80)
     → No son informativos, pero tampoco dañan

  6. Renombrar columnas según mapeo del proyecto

  7. Castear categóricas a dtype 'category'

  8. NO escalar euribor3m ni cons.price.idx
     → El CSV con separador `;` ya tiene valores reales
''')

# Impacto final
rows_to_remove = 12 + 990  # duplicados + housing/loan unknowns
rows_final = n_rows - rows_to_remove
print(f'  📊 IMPACTO:')
print(f'     Filas originales:  {n_rows:>7,}')
print(f'     Filas a eliminar:  {rows_to_remove:>7,} ({rows_to_remove/n_rows*100:.2f}%)')
print(f'     Filas finales:     {rows_final:>7,}')
print(f'     Unknowns restantes: default={8597-0:,}, education={1731-0:,}, job={330-0:,}, marital={80-0:,}')
print(f'     → Tratados como categorías legítimas, NO como NaN')
print()
print('=' * 90)
print('  FIN DEL ANÁLISIS — DATOS NO MODIFICADOS')
print('=' * 90)

In [ ]:
# Limpieza de variable auxiliar
# (No se guarda nada: este notebook es puramente analítico)
if 'y_num' in df.columns:
    df.drop(columns=['y_num', 'age_bin'], inplace=True, errors='ignore')
    print('✅ Variables auxiliares eliminadas del DataFrame en memoria.')
    print('✅ El CSV original NO fue modificado.')